# Data Preprocessing
Fariha Adil, 2375026

## About
The goal of this analysis is to load and preprocess the raw DBLP dataset for use in downstream analysis. The cleaned dataset is saved for use by all group members.

## Tasks
- Load the raw DBLP dataset from the four JSON files
- Handle missing data across fields
- Filter sparse venues and normalize author names
- Document dataset statistics
- Save the cleaned data for use by teammates

## Motivations
- The raw dataset contains noise such as missing fields and venues with very few papers that could skew analysis
- A shared cleaned dataset ensures consistency across all three analyses

## Challenges
- Author disambiguation: the same name can refer to different authors, making normalization imprecise

In [1]:
"""Import Dependencies"""
import json
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
"""Globals/Constants"""
DATA_PATH = './dblp_ref/'
MIN_VENUE_COUNT = 10

In [3]:
"""Helper/Utility Functions"""
def read_json_lines(file_path):
    """Reads the JSON file at the given path and yields each line (record)."""
    with open(file_path) as json_file:
        for line in json_file:
            yield json.loads(line)

def normalize_author(name):
    return name.strip().lower() if isinstance(name, str) else None

def is_missing(val):
    return val is None or (isinstance(val, float) and np.isnan(val)) or val == '' or val == []

In [4]:
"""Load Data"""
frames = []
for file_index in range(4):
    file_name = f'{DATA_PATH}dblp-ref-{file_index}.json'
    frames.append(pd.DataFrame(read_json_lines(file_name)))

papers = pd.concat(frames, ignore_index = True)
papers.head()

,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[Makoto Satoh, Ryo Muramatsu, Mizue Kayama, Ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[Gareth Beale, Graeme Earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[Altaf Hossain, Faisal Zaman, Mohammed Nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[Jea-Bum Park, Byungmok Kim, Jian Shen, Sun-Yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[Giovanna Guerrini, Isabella Merlo]",2,NaN,Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266


In [5]:
"""Preprocessing: Finding Missing Data in Dataset"""
total = len(papers)

missing_venue = (papers['venue'] == '').sum()
missing_references = papers['references'].apply(is_missing).sum()
missing_abstract = papers['abstract'].apply(is_missing).sum()
missing_authors = papers['authors'].apply(is_missing).sum()

print(f"Total papers: {total:,}")
print(f"Missing venue: {missing_venue:,} ({missing_venue/total*100:.0f}%)")
print(f"Missing references: {missing_references:,} ({missing_references/total*100:.2f}%)")
print(f"Missing abstract: {missing_abstract:,} ({missing_abstract/total*100:.2f}%)")
print(f"Missing authors: {missing_authors:,} ({missing_authors/total*100:.4f}%)")

citation_counts = papers['n_citation']
print(f"\nCitation Count Statistics:")
print(f"Mean: {citation_counts.mean():.2f}")
print(f"Median: {citation_counts.median():.0f}")
print(f"Max: {citation_counts.max():,}")
print(f"Zero citations: {(citation_counts == 0).sum():,} ({(citation_counts == 0).sum()/total*100:.2f}%)")

Total papers: 3,079,007
Missing venue: 506,699 (16%)
Missing references: 574,884 (18.67%)
Missing abstract: 530,475 (17.23%)
Missing authors: 4 (0.0001%)

Citation Count Statistics:
Mean: 35.22
Median: 11
Max: 73,362
Zero citations: 718,250 (23.33%)


In [9]:
"""Preprocessing: Dataset Statistics"""
print("Year Distribution")
pd.set_option('display.float_format', '{:.0f}'.format)
print(papers['year'].describe())
pd.reset_option('display.float_format')

print(f"  Before 1990: {(papers['year'] < 1990).sum():,}")
print(f"  1990–1999: {((papers['year'] >= 1990) & (papers['year'] < 2000)).sum():,}")
print(f"  2000–2009: {((papers['year'] >= 2000) & (papers['year'] < 2010)).sum():,}")
print(f"  2010–2018: {((papers['year'] >= 2010) & (papers['year'] <= 2018)).sum():,}")

print("\nCitation Count (n_citation)")
pd.set_option('display.float_format', '{:.2f}'.format)
print(papers['n_citation'].describe())
pd.reset_option('display.float_format')
print(f"  Papers with 0 citations: {(papers['n_citation'] == 0).sum():,}")
print(f"  Papers with 1000+: {(papers['n_citation'] >= 1000).sum():,}")
print(f"  Top cited paper: {papers['n_citation'].max():,} citations")
print(papers.nlargest(1, 'n_citation')[['title', 'year', 'n_citation']].to_string())


print("# of References per Paper")
ref_lengths = papers['references'].apply(lambda x: len(x) if isinstance(x, list) else 0)
pd.set_option('display.float_format', '{:.2f}'.format)
print(ref_lengths.describe())
pd.reset_option('display.float_format')

print("Authors per Paper")
author_counts = papers['authors'].apply(lambda x: len(x) if isinstance(x, list) else 0)
pd.set_option('display.float_format', '{:.2f}'.format)
print(author_counts.describe())
pd.reset_option('display.float_format')

idx = author_counts.idxmax()
print(papers.loc[idx, 'title'])
print(f"Author count: {author_counts[idx]}")
print(papers.loc[idx, 'authors'])

print("Most Prolific Authors")
all_authors = [a for authors in papers['authors'].dropna() for a in authors if isinstance(authors, list)]
top_authors = Counter(all_authors).most_common(10)
for author, count in top_authors:
    print(f"  {author}: {count} papers")

print("Venue Coverage")
venue_counts = papers['venue'].value_counts()
print(f"  Unique venues: {venue_counts.shape[0]:,}")
print(f"  Venues with 1 paper: {(venue_counts == 1).sum():,}")
print(f"  Venues with 10+ papers: {(venue_counts >= 10).sum():,}")
print(f"  Venues with 100+ papers: {(venue_counts >= 100).sum():,}")
print("\n  Top 3 venues:")
print(venue_counts.head(5).to_string())

Year Distribution
count   3079007
mean       2008
std           8
min        1936
25%        2004
50%        2010
75%        2013
max        2018
Name: year, dtype: float64
  Before 1990: 100,995
  1990–1999: 301,547
  2000–2009: 1,110,810
  2010–2018: 1,565,655

Citation Count (n_citation)
count   3079007.00
mean         35.22
std         157.70
min           0.00
25%           1.00
50%          11.00
75%          50.00
max       73362.00
Name: n_citation, dtype: float64
  Papers with 0 citations: 718,250
  Papers with 1000+: 4,988
  Top cited paper: 73,362 citations
                                                                  title  year  n_citation
551841  Genetic Algorithms in Search, Optimization and Machine Learning  1989       73362
# of References per Paper
count   3079007.00
mean          8.17
std           9.71
min           0.00
25%           1.00
50%           6.00
75%          12.00
max        1532.00
Name: references, dtype: float64
Authors per Paper
count   3079007.

In [7]:
"""Preprocessing: Filtering and Cleaning"""
cleaned = papers.copy()

cleaned = cleaned.dropna(subset = ['title', 'authors', 'id'])
print(f"After dropping incomplete rows: {len(cleaned):,}")

cleaned['authors'] = cleaned['authors'].apply(
    lambda author_list: [normalize_author(a) for a in author_list] 
    if isinstance(author_list, list) else []
)

cleaned['venue'] = cleaned['venue'].fillna('')
venue_counts = cleaned['venue'].value_counts()
valid_venues = venue_counts[venue_counts >= MIN_VENUE_COUNT].index
cleaned['venue'] = cleaned['venue'].apply(
    lambda v: v if v in valid_venues else 'Other')
print(f"Venues after filtering: {cleaned['venue'].nunique():,}")

cleaned['references'] = cleaned['references'].apply(
    lambda refs: refs if isinstance(refs, list) else []
)

print(f"\nFinal cleaned dataset: {len(cleaned):,} papers")
cleaned.head()

After dropping incomplete rows: 3,079,003
Venues after filtering: 3,116

Final cleaned dataset: 3,079,003 papers


,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[makoto satoh, ryo muramatsu, mizue kayama, ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[gareth beale, graeme earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[altaf hossain, faisal zaman, mohammed nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[jea-bum park, byungmok kim, jian shen, sun-yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[giovanna guerrini, isabella merlo]",2,[],Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266


In [ ]:
"""Saves Cleaned Data for Teammates"""
cleaned.to_pickle('cleaned_dataset.pkl')